# Диалоговые подсказки для рекомендаций мест

Пользовательские реплики часто зависят от всей истории разговора. Здесь из контекста диалога генерируются короткие варианты продолжения и самостоятельные поисковые запросы, которые можно передать поиску без предыдущих сообщений.

## 1. Окружение

Здесь подключаются библиотеки и фиксируется случайность. Внешние модели и веса не используются.

In [ ]:
%pip install -q "pandas==3.1.0" "numpy==2.5.4" "scikit-learn==1.9.1" "seaborn==0.13.2" "matplotlib==3.11.2" "tqdm==4.67.1"

In [ ]:
import platform
import random
from importlib.metadata import version

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", platform.python_version())
for package in ["pandas", "numpy", "scikit-learn"]:
    print(f"{package}: {version(package)}")

In [ ]:
GENERATION_METHOD = "candidate-library-v1"
SUGGESTIONS_PER_DIALOGUE = 4
MAX_SUGGESTION_WORDS = 12
SEMANTIC_DUPLICATE_THRESHOLD = 0.72

## 2. Диалоги

Набор состоит из 48 диалогов о местах: по 12 примеров первичного запроса, уточнения, отказа и изменения направления. Сами данные лежат в `dialogues.json`; ноутбук только читает и проверяет их.

In [ ]:
import json

SCENARIO_NAMES = {
    "initial": "Первичная рекомендация",
    "refinement": "Уточнение предпочтений",
    "rejection": "Отказ от рекомендаций",
    "direction_change": "Изменение направления",
}

DIRECTION_FALLBACK = [
    "atmosphere", "location", "budget", "features", "company",
    "time", "category", "activity", "cuisine", "alternative",
]


In [ ]:
from pathlib import Path

DIALOGUES_PATH = Path("dialogues.json")
PREFERENCES_PATH = Path("preference_taxonomy.csv")
CANDIDATES_PATH = Path("suggestion_candidates.csv")

required_files = [DIALOGUES_PATH, PREFERENCES_PATH, CANDIDATES_PATH]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing files: " + ", ".join(missing_files))

dialogues = json.loads(DIALOGUES_PATH.read_text())
preferences = pd.read_csv(PREFERENCES_PATH).set_index("key")["text"]
candidates = pd.read_csv(CANDIDATES_PATH).fillna("")
candidates["blocked_by"] = candidates["blocked_by"].map(
    lambda value: frozenset(filter(None, str(value).split("|")))
)

required_fields = {
    "dialogue_id", "scenario", "messages",
    "known_preferences", "forbidden_preferences", "missing_slots",
}
preference_keys = set(preferences.index)
directions = set(candidates["direction"])
invalid_dialogues = []
for dialogue in dialogues:
    keys = set(dialogue) if isinstance(dialogue, dict) else set()
    messages = dialogue.get("messages", []) if isinstance(dialogue, dict) else []
    state = [
        *dialogue.get("known_preferences", []),
        *dialogue.get("forbidden_preferences", []),
    ] if isinstance(dialogue, dict) else []
    valid = (
        required_fields <= keys
        and dialogue.get("scenario") in SCENARIO_NAMES
        and bool(messages)
        and all(message.get("role") in {"user", "assistant"} and message.get("content") for message in messages)
        and set(state) <= preference_keys
        and set(dialogue.get("missing_slots", [])) <= directions
        and not set(dialogue.get("known_preferences", [])) & set(dialogue.get("forbidden_preferences", []))
    )
    if not valid:
        invalid_dialogues.append(dialogue.get("dialogue_id", "unknown") if isinstance(dialogue, dict) else "unknown")

ids = [dialogue["dialogue_id"] for dialogue in dialogues]
candidate_schema = {"candidate_id", "direction", "suggestion", "query_value", "blocked_by"}
if invalid_dialogues or len(ids) != len(set(ids)) or not candidate_schema <= set(candidates.columns):
    raise ValueError(f"Invalid input data; dialogues={invalid_dialogues}")

dialogues_df = pd.DataFrame(dialogues)
dialogues_df["scenario_name"] = dialogues_df["scenario"].map(SCENARIO_NAMES)
print(f"Dialogues loaded: {len(dialogues_df)}")
display(dialogues_df[["dialogue_id", "scenario_name", "known_preferences", "forbidden_preferences", "missing_slots"]].head())


In [ ]:
scenario_summary = dialogues_df.groupby("scenario_name").agg(
    dialogues=("dialogue_id", "count"),
    known_preferences=("known_preferences", lambda values: np.mean([len(value) for value in values])),
    forbidden_preferences=("forbidden_preferences", lambda values: np.mean([len(value) for value in values])),
)
scenario_summary

## 3. Генерация

Кандидаты подсказок хранятся отдельно от кода. Генератор выбирает разные направления, отбрасывает варианты, которые конфликтуют с запретами, и собирает самостоятельный поисковый запрос из известных условий.


In [ ]:
print("Generation method:", GENERATION_METHOD)
print("External model weights: none")

In [ ]:
import hashlib

def choose_directions(dialogue):
    ordered = [*dialogue["missing_slots"], "alternative", *DIRECTION_FALLBACK]
    return list(dict.fromkeys(ordered))[:SUGGESTIONS_PER_DIALOGUE]

def generate_suggestions(dialogue):
    blocked = set(dialogue["forbidden_preferences"])
    known_text = preferences.reindex(dialogue["known_preferences"]).tolist()
    forbidden_text = preferences.reindex(dialogue["forbidden_preferences"]).tolist()
    generated = []
    for direction in choose_directions(dialogue):
        pool = candidates[
            candidates["direction"].eq(direction)
            & candidates["blocked_by"].map(blocked.isdisjoint)
        ].reset_index(drop=True)
        if pool.empty:
            raise RuntimeError(f"No candidate for {dialogue['dialogue_id']} and {direction}")
        seed = f"{dialogue.get('seed_id', dialogue['dialogue_id'])}:{direction}".encode()
        index = int.from_bytes(hashlib.blake2b(seed, digest_size=4).digest(), "big") % len(pool)
        candidate = pool.iloc[index]
        query_parts = list(dict.fromkeys([*known_text, candidate["query_value"]]))
        query = "Место: " + ", ".join(query_parts)
        if forbidden_text:
            query += "; исключить: " + ", ".join(forbidden_text)
        generated.append({
            "candidate_id": candidate["candidate_id"],
            "suggestion": candidate["suggestion"],
            "query": query,
            "direction": direction,
            "blocked_by": "|".join(sorted(candidate["blocked_by"])),
        })
    return generated


In [ ]:
from tqdm.auto import tqdm

generated_rows = []
for dialogue in tqdm(dialogues, desc="Generating suggestions"):
    for rank, item in enumerate(generate_suggestions(dialogue), start=1):
        generated_rows.append({
            "dialogue_id": dialogue["dialogue_id"],
            "scenario": dialogue["scenario"],
            "rank": rank,
            **item,
        })

generated_df = pd.DataFrame(generated_rows)
generated_df.to_csv("generated_suggestions.csv", index=False)
display(generated_df[["dialogue_id", "rank", "suggestion", "query", "direction"]].head(8))


## 4. Автоматическая оценка

Оценка проверяет длину подсказок, сохранение известных условий, отсутствие конфликтов, разнообразие направлений и смысловые повторы внутри одного диалога.


In [ ]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).lower().replace("ё", "е")
    return " ".join(re.sub(r"[^a-zа-я0-9]+", " ", text).split())

def preference_coverage(query, keys):
    terms = preferences.reindex(keys).dropna().map(normalize_text)
    normalized_query = normalize_text(query)
    return float(terms.map(lambda term: term in normalized_query).mean()) if len(terms) else 1.0

dialogue_lookup = {item["dialogue_id"]: item for item in dialogues}
evaluation_rows = []
for row in generated_df.itertuples(index=False):
    dialogue = dialogue_lookup[row.dialogue_id]
    blocked = set(filter(None, row.blocked_by.split("|")))
    evaluation_rows.append({
        "dialogue_id": row.dialogue_id,
        "rank": row.rank,
        "word_count": len(normalize_text(row.suggestion).split()),
        "short_enough": len(normalize_text(row.suggestion).split()) <= MAX_SUGGESTION_WORDS,
        "known_coverage": preference_coverage(row.query, dialogue["known_preferences"]),
        "forbidden_conflict": not blocked.isdisjoint(dialogue["forbidden_preferences"]),
        "adds_missing_slot": row.direction in dialogue["missing_slots"] or row.direction == "alternative",
    })

evaluation_df = pd.DataFrame(evaluation_rows)
generated_df = generated_df.merge(evaluation_df, on=["dialogue_id", "rank"], validate="one_to_one")
generated_df.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

duplicate_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True)
suggestion_vectors = duplicate_vectorizer.fit_transform(generated_df["suggestion"])

generated_df["semantic_duplicate"] = False
generated_df["max_pair_similarity"] = 0.0
for indices in generated_df.groupby("dialogue_id").groups.values():
    indices = list(indices)
    similarities = cosine_similarity(suggestion_vectors[indices])
    np.fill_diagonal(similarities, -1)
    maxima = similarities.max(axis=1)
    generated_df.loc[indices, "max_pair_similarity"] = maxima
    generated_df.loc[indices, "semantic_duplicate"] = maxima >= SEMANTIC_DUPLICATE_THRESHOLD

generated_df["exact_duplicate"] = generated_df.groupby("dialogue_id")["suggestion"].transform(
    lambda values: values.map(normalize_text).duplicated(keep=False)
)
generated_df["direction_diversity"] = generated_df.groupby("dialogue_id")["direction"].transform("nunique") / SUGGESTIONS_PER_DIALOGUE
generated_df["passes_checks"] = (
    generated_df["short_enough"]
    & generated_df["known_coverage"].eq(1)
    & ~generated_df["forbidden_conflict"]
    & generated_df["adds_missing_slot"]
    & ~generated_df["semantic_duplicate"]
    & generated_df["direction_diversity"].eq(1)
)
generated_df.to_csv("suggestions_with_metrics.csv", index=False)
display(generated_df[["dialogue_id", "suggestion", "known_coverage", "forbidden_conflict", "adds_missing_slot", "semantic_duplicate", "passes_checks"]].head(12))


## 5. Анализ результатов

Сводные таблицы показывают метрики для всей выборки и отдельно для каждого типа сценария. Ниже приведены результаты генератора и контрольные примеры типичных ошибок.


In [ ]:
metric_columns = [
    "short_enough", "known_coverage", "forbidden_conflict",
    "adds_missing_slot", "semantic_duplicate", "direction_diversity", "passes_checks",
]
overall_metrics = generated_df[metric_columns].mean().rename("value").to_frame()
scenario_metrics = generated_df.groupby("scenario")[metric_columns].mean()
display(overall_metrics.round(3))
display(scenario_metrics.round(3))

diversity_metrics = pd.Series({
    "mean_words": generated_df["word_count"].mean(),
    "p95_words": generated_df["word_count"].quantile(0.95),
    "unique_text_share": generated_df["suggestion"].nunique() / len(generated_df),
    "candidate_coverage": generated_df["candidate_id"].nunique() / len(candidates),
    "mean_max_pair_similarity": generated_df["max_pair_similarity"].mean(),
    "directions_used": generated_df["direction"].nunique(),
}).rename("value").to_frame()
display(diversity_metrics.round(3))

direction_metrics = generated_df.groupby("direction").agg(
    suggestions=("suggestion", "size"),
    unique_texts=("suggestion", "nunique"),
    mean_words=("word_count", "mean"),
    duplicate_rate=("semantic_duplicate", "mean"),
).sort_values("suggestions", ascending=False)
display(direction_metrics.round(3))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
scenario_counts = dialogues_df["scenario_name"].value_counts()
sns.barplot(x=scenario_counts.values, y=scenario_counts.index, ax=axes[0], color="steelblue")
axes[0].set_title("Dialogue scenarios")
axes[0].set_xlabel("Dialogues")
axes[0].set_ylabel("")

plot_metrics = overall_metrics.loc[["known_coverage", "adds_missing_slot", "direction_diversity", "passes_checks"]].reset_index()
sns.barplot(data=plot_metrics, x="value", y="index", ax=axes[1], color="seagreen")
axes[1].set_xlim(0, 1)
axes[1].set_title("Suggestion quality")
axes[1].set_xlabel("Share")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
heatmap_metrics = scenario_metrics[[
    "known_coverage", "adds_missing_slot", "forbidden_conflict", "semantic_duplicate", "direction_diversity"
]]
plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_metrics, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Metrics by scenario")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
good_examples = generated_df.loc[generated_df["passes_checks"], [
    "dialogue_id", "suggestion", "query", "direction"
]].groupby("direction", group_keys=False).head(1).head(8)

control_examples = pd.DataFrame([
    {
        "type": "context conflict",
        "suggestion": "Давай шумный бар",
        "why": "Возвращает громкое место после прямого отказа от него",
    },
    {
        "type": "too general",
        "suggestion": "Покажи что-нибудь ещё",
        "why": "Не добавляет ни одного нового условия поиска",
    },
    {
        "type": "semantic duplicate",
        "suggestion": "Найди недорогое место / Покажи бюджетный вариант",
        "why": "Две формулировки выражают одно намерение",
    },
    {
        "type": "not standalone",
        "suggestion": "Хочу вечером",
        "why": "Без истории непонятны тип места и уже известные условия",
    },
    {
        "type": "off topic",
        "suggestion": "Посоветуй новый сериал",
        "why": "Уводит разговор от поиска мест",
    },
])

print("Good examples")
display(good_examples)
print("Controlled failure examples")
display(control_examples)

## 6. Эксперименты с контекстом

В четырёх диалогах меняется одно условие: бюджет, компания, погода или район. Сравнение показывает, сохранились ли старые предпочтения и появились ли новые условия в поисковом запросе.


In [ ]:
experiment_specs = [
    {
        "experiment_id": "budget",
        "source_id": "initial_01",
        "message": "Бюджет можно увеличить, хочется место высокого уровня.",
        "add_known": ["premium"],
        "add_forbidden": ["low_budget"],
        "remove_missing": ["budget"],
    },
    {
        "experiment_id": "company",
        "source_id": "initial_11",
        "message": "Теперь со мной будут друзья.",
        "add_known": ["friends"],
        "remove_known": ["alone"],
        "add_forbidden": ["alone"],
        "remove_missing": ["company"],
    },
    {
        "experiment_id": "weather",
        "source_id": "initial_02",
        "message": "На улице дождь, нужны варианты в помещении.",
        "add_known": ["indoor"],
        "add_forbidden": ["outdoor"],
        "remove_missing": ["category"],
    },
    {
        "experiment_id": "location",
        "source_id": "initial_08",
        "message": "В центре уже всё посмотрел, хочу другой район.",
        "add_known": ["new_area"],
        "add_forbidden": ["center"],
        "remove_missing": ["location"],
    },
]

dialogue_lookup = {item["dialogue_id"]: item for item in dialogues}
context_variants = []
for spec in experiment_specs:
    source = dialogue_lookup[spec["source_id"]]
    variant = {
        **source,
        "dialogue_id": f"variant_{spec['experiment_id']}",
        "seed_id": source["dialogue_id"],
        "messages": [
            *source["messages"],
            {"role": "user", "content": spec["message"]},
            {"role": "assistant", "content": "Учту изменение. Что ещё важно?"},
        ],
        "known_preferences": list(dict.fromkeys([
            *[value for value in source["known_preferences"] if value not in spec.get("remove_known", [])],
            *spec["add_known"],
        ])),
        "forbidden_preferences": list(dict.fromkeys([*source["forbidden_preferences"], *spec["add_forbidden"]])),
        "missing_slots": [slot for slot in source["missing_slots"] if slot not in spec["remove_missing"]],
    }
    context_variants.append(variant)

variants_df = pd.DataFrame(context_variants)
variants_df[["dialogue_id", "known_preferences", "forbidden_preferences", "missing_slots"]]

In [ ]:
experiment_rows = []
for spec, variant in tqdm(zip(experiment_specs, context_variants), total=len(context_variants), desc="Running context experiments"):
    before_rows = generated_df[generated_df["dialogue_id"] == spec["source_id"]]
    for row in before_rows.itertuples(index=False):
        experiment_rows.append({
            "experiment_id": spec["experiment_id"],
            "version": "before",
            "rank": row.rank,
            "direction": row.direction,
            "suggestion": row.suggestion,
            "query": row.query,
            "blocked_by": row.blocked_by,
        })
    for rank, item in enumerate(generate_suggestions(variant), start=1):
        experiment_rows.append({
            "experiment_id": spec["experiment_id"],
            "version": "after",
            "rank": rank,
            **item,
        })

context_results_df = pd.DataFrame(experiment_rows)
context_results_df.to_csv("context_experiments.csv", index=False)
context_results_df


In [ ]:
experiment_evaluation = []
for spec, variant in zip(experiment_specs, context_variants):
    rows = context_results_df[context_results_df["experiment_id"] == spec["experiment_id"]]
    for row in rows.itertuples(index=False):
        blocked = set(filter(None, row.blocked_by.split("|")))
        experiment_evaluation.append({
            "experiment_id": spec["experiment_id"],
            "version": row.version,
            "rank": row.rank,
            "changed_preference_coverage": preference_coverage(row.query, spec["add_known"]) if row.version == "after" else np.nan,
            "retained_preference_coverage": preference_coverage(row.query, variant["known_preferences"]) if row.version == "after" else np.nan,
            "forbidden_conflict": not blocked.isdisjoint(variant["forbidden_preferences"]),
        })

experiment_evaluation_df = pd.DataFrame(experiment_evaluation)
after_summary = experiment_evaluation_df[experiment_evaluation_df["version"] == "after"].groupby("experiment_id").agg(
    changed_preference_coverage=("changed_preference_coverage", "mean"),
    retained_preference_coverage=("retained_preference_coverage", "mean"),
    forbidden_conflict=("forbidden_conflict", "mean"),
)
display(after_summary.round(3))

before = context_results_df[context_results_df["version"] == "before"]
after = context_results_df[context_results_df["version"] == "after"]
paired = before.merge(after, on=["experiment_id", "direction"], suffixes=("_before", "_after"), validate="one_to_one")
paired["suggestion_changed"] = paired["suggestion_before"] != paired["suggestion_after"]
paired["query_changed"] = paired["query_before"] != paired["query_after"]
change_summary = paired.groupby("experiment_id").agg(
    common_directions=("direction", "size"),
    changed_suggestions=("suggestion_changed", "sum"),
    changed_queries=("query_changed", "sum"),
)
display(change_summary)
display(paired[["experiment_id", "direction", "suggestion_before", "suggestion_after", "query_before", "query_after"]])


## 7. Выводы

Генератор работает без внешней модели. Его сильная сторона — контролируемые ограничения; основное ограничение — небольшой набор формулировок в библиотеке кандидатов.


In [ ]:
print(f"Suggestions evaluated: {len(generated_df)}")
print(f"Overall pass rate: {generated_df['passes_checks'].mean():.3f}")
print(f"Known preference coverage: {generated_df['known_coverage'].mean():.3f}")
print(f"Forbidden conflict rate: {generated_df['forbidden_conflict'].mean():.3f}")
print(f"Semantic duplicate rate: {generated_df['semantic_duplicate'].mean():.3f}")
print(f"Unique suggestion texts: {generated_df['suggestion'].nunique()} of {len(generated_df)}")
display(scenario_metrics.round(3))
